In [ ]:
import sys
from pathlib import Path

# Add src/ to Python path for local module imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

# Preliminari

In [1]:
import os, re
import torch
import json
import copy
#from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset
#from unsloth import FastLanguageModel
from pathlib import Path
#from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoTokenizer, AutoModelForCausalLM

#from unsloth.chat_templates import train_on_responses_only
#from unsloth.chat_templates import get_chat_template
from dotenv import load_dotenv
from transformers import TextStreamer
from transformers import BitsAndBytesConfig

from huggingface_hub import login

import pandas as pd
from pathlib import Path

from pydantic import BaseModel

import os

import constants
from constants import RectalCancerStagingData

from prompting_utils import create_system_prompt

from model_utils import from_series_to_basemodel

from tqdm import tqdm

from pathlib import Path

import json
import outlines

import torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True
from peft import PeftModel


In [2]:
def get_device():
    print(f'{torch.cuda.is_available() = }')  # True se la GPU è disponibile
    print(f'{torch.cuda.device_count() = }')  # Numero di GPU disponibili
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Parametri

In [3]:
# Log in huggingface
load_dotenv()  # Load environment variables from .env file
hf_token = os.getenv('HF_TOKEN')
login(hf_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Parametri
base_dir = Path.cwd().parent
SYSTEM_PROMPT_FILE_NAME = constants.SYSTEM_PROMPT_SHORT

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER = constants.LLAMA_3_2_1B_TUNED
ADAPTER = ''

#model_name = "meta-llama/Llama-3.1-8B-Instruct"
#model_name = "meta-llama/Llama-3.2-1B-Instruct"
#model_name = "meta-llama/Llama-3.1-3B-Instruct"
#model_name = "sapienzanlp/Minerva-7B-instruct-v1.0"
#model_name = "mistralai/Mistral-7B-Instruct-v0.3"

RESULTS_FILE_NAME = 'new_results_' + 'llama-3b' + '.jsonl'

PYDANTIC_MODEL = constants.RectalCancerStagingData

dtype = torch.float16 # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

USE_OUTLINES = True

#Carica system prompt da file
system_prompt_path = base_dir / "data" / "prompts" / SYSTEM_PROMPT_FILE_NAME
system_prompt = create_system_prompt(system_prompt_path, PYDANTIC_MODEL)

In [5]:
if True:
    print(system_prompt)

Sei un assistente medico radiologico esperto nella stadiazione del carcinoma rettale tramite RM.
Estrai informazioni strutturate dal referto RM fornito e restituisci esclusivamente un oggetto JSON valido conforme allo schema seguente.

<schema>
{
  "morfologia": "solido_polipoide | solido_anulare | mucinoso",
  "ore_inizio": "int | null",
  "ore_fine": "int | null",
  "spessore_parietale": "int | null",
  "estensione_cranio_caudale": "int | null",
  "distanza_oai": "int | null",
  "posizione": {
    "basso": "no | si",
    "medio": "no | si",
    "alto": "no | si",
    "giunzione": "no | si"
  },
  "riflessione_peritoneale_anteriore": "sotto | cavallo | non_descritto",
  "infiltrazione_tessuto_adiposo": "no | si_5mm | si_5mm_plus",
  "infiltrazione_sfinteri": "no | si",
  "infiltrazione_organi_extra": "no | si",
  "infiltrazione_organi_dettagli": {
    "muscolo_elevatore": "no | si",
    "pavimento_pelvico": "no | si",
    "altro": "no | si"
  },
  "coinvolgimento_riflessione_peritonea

# Load Data

In [6]:
# Carichiamo i nostri file csv
file_names = {
    'train': constants.TRAIN_SPLIT_FILE_NAME,
    'validation': constants.VALIDATION_SPLIT_FILE_NAME,
    'test': constants.TEST_SPLIT_FILE_NAME,
}

paths = {split: Path('../data/' + file_name) for split, file_name in file_names.items()}

data = dict()
for split, path in paths.items():
    data[split] = pd.read_csv(path)

validation_data, test_data = data['validation'], data['test']
train_data = data['train']

################################
# Convert float columns to Int64
################################
float_cols = test_data.select_dtypes("float").columns
for col in float_cols:
    test_data[col] = test_data[col].round().astype("Int64")
    validation_data[col] = validation_data[col].round().astype("Int64")

# Check duplicatest
assert set(test_data.id) & set(validation_data.id) == set() & set(train_data.id) == set(), "There are overlapping IDs between test and validation sets!"

print(f"{len(train_data) = }")
print(f"{len(test_data) = }")
print(f"{len(validation_data) = }")

len(train_data) = 187
len(test_data) = 65
len(validation_data) = 63


# Load model and tokenizer

In [7]:
device = get_device()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=load_in_4bit,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

hf_model = AutoModelForCausalLM.from_pretrained(MODEL,
                                                device_map="cuda",
                                                quantization_config=bnb_config)
if ADAPTER != '':
    model = PeftModel.from_pretrained(hf_model, ADAPTER)
    hf_model.generation_config.pad_token_id = tokenizer.eos_token_id
    if USE_OUTLINES:
        model = outlines.from_transformers(model, tokenizer)
else:
    hf_model.generation_config.pad_token_id = tokenizer.eos_token_id
    if USE_OUTLINES:
        model = outlines.from_transformers(hf_model, tokenizer)
    else:
        model = hf_model

print(f'Memoria usata dal modello {MODEL}')
print(f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB allocati")
print(f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB riservati")

torch.cuda.is_available() = True
torch.cuda.device_count() = 1
NVIDIA GeForce GTX 1060 6GB


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]


Memoria usata dal modello meta-llama/Llama-3.2-3B-Instruct
2.23 GB allocati
2.25 GB riservati


In [8]:
def extract_data_from_report(model, report_text, system_prompt=system_prompt,
                             tokenizer=tokenizer, output_format=PYDANTIC_MODEL,
                             return_basemodel=False, use_outlines=USE_OUTLINES):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": report_text},
    ]

    if use_outlines:
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        json_str = model(prompt, output_type=output_format, max_new_tokens=2048, do_sample=False)
    else:
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        prompt += "{"
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        outputs = model.generate(**inputs, max_new_tokens=2048, do_sample=False)
        json_str = "{" + tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    if return_basemodel:
        try:
            return output_format.model_validate_json(json_str)
        except Exception:
            return None
    return json_str

# Inference

In [9]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [10]:
print(MODEL)
RET_BASE_MODEL = True
print(f'{RET_BASE_MODEL = }')
print(f'{USE_OUTLINES = }')
df = pd.concat([train_data, validation_data, test_data], ignore_index=True)
#df = pd.concat([validation_data, test_data], ignore_index=True)
ids = []
actual = []
predicted = []
splits = []
for i in tqdm(range(df.shape[0])):
    row = df.iloc[i]
    splits.append(row['split'])
    output = extract_data_from_report(model=model, report_text=row[constants.REPORT_COLUMN_NAME], return_basemodel=RET_BASE_MODEL)
    real = from_series_to_basemodel(row, PYDANTIC_MODEL)
    ids.append(row[constants.ID_COMULM_NAME])
    if RET_BASE_MODEL:
        actual.append(real.model_dump(mode='json'))
    else:
        actual.append(real.model_dump_json())
    if output is None:
        predicted.append('no output')
    else:
        if not RET_BASE_MODEL:
            predicted.append(output)
        else: predicted.append(output.model_dump(mode='json'))
    torch.cuda.empty_cache()

meta-llama/Llama-3.2-3B-Instruct
RET_BASE_MODEL = True
USE_OUTLINES = True


  0%|          | 0/315 [00:00<?, ?it/s]c:\Users\lucat\PythonRepositories\PRIN\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\lucat\PythonRepositories\PRIN\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
  0%|          | 0/315 [01:16<?, ?it/s]


KeyboardInterrupt: 

In [11]:
actual[0]

{'morfologia': 'solido_anulare',
 'ore_inizio': 12,
 'ore_fine': 12,
 'spessore_parietale': None,
 'estensione_cranio_caudale': 100,
 'distanza_oai': 70,
 'posizione': {'basso': 'no', 'medio': 'no', 'alto': 'si', 'giunzione': 'si'},
 'riflessione_peritoneale_anteriore': 'cavallo',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': {'muscolo_elevatore': 'no',
  'pavimento_pelvico': 'no',
  'altro': 'no'},
 'coinvolgimento_riflessione_peritoneale': 'si',
 'coinvolgimento_fascia_mesorettale': 'si',
 'numero_linfonodi_non_conosciuto': 'non_conosciuto',
 'linfonodi_sospetti': 0,
 'sedi_linfonodi': {'mesorettali': 'si',
  'rettali_superiori': 'si',
  'otturatori': 'no',
  'iliaci': 'no',
  'altro': 'no'},
 'depositi_tumorali': 'si',
 'emvi': 'no',
 'stadio_T': 'T4a',
 'stadio_N': 'N+',
 'mrf': 'si',
 'metastasi': 'MX'}

In [12]:
type(actual[1])

dict

In [13]:
results_dicts = []
assert len(ids) == len(actual) == len(predicted)
for id, act, pred, split in zip(ids, actual, predicted, splits):
    results_dicts.append(
        {
            'model': MODEL if ADAPTER == "" else ADAPTER,
            'split': split,
            'id': int(id),
            'actual': act,
            'prediction': pred,
            'type': 'basemodel' if RET_BASE_MODEL else 'str'
        }
    )
# Salvataggio
with open(base_dir / 'data' / 'inference' / RESULTS_FILE_NAME, 'w', encoding='utf-8') as f:
    for r in results_dicts:
        f.write(json.dumps(r) + '\n')

In [25]:
#new_pred = []
#new_actual = []
#for p, a in zip(predicted, actual):
#    new_pred.append(PYDANTIC_MODEL.model_validate_json(a).model_dump(mode='json'))
#    new_actual.append(PYDANTIC_MODEL.model_validate_json(a).model_dump(mode='json'))
#RET_BASE_MODEL = True

In [24]:
#predicted = new_pred
#actual = new_actual

# Data preparation

I dati in `raw_json_data` sono salvati con la seguente struttura:
```
{
    "conversations": [
        {
            "messages": [
                {"role": "system", "content": <contenuto di sistema>},
                {"role": "user", "content": <contenuto di user>},
                {"role": "assistant", "content": <contenuto di assistant>}
            ]
        },
        ...
    ]
}
```

## Creazione dataset corretto

Trasformiamo `<contenuto di assistant>` in una stringa.

In [10]:
training_data = copy.deepcopy(raw_json_data)

for conversation in training_data['conversations']:
    for message in conversation['messages']:
        if message['role'] == 'assistant':
            message['content'] = json.dumps(message['content'])

Ora possiamo applicare il chat template, dato che il contenuto è una stringa

In [11]:
# Creiamo ora un "Huggingface dataset" con i nostri dati
dataset = Dataset.from_dict(training_data)
print(dataset)
display(dataset.features)

Dataset({
    features: ['conversations'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]}}

## Creazione nuova colonna

Noi vogliamo aggiungere la colonna "text" al nostro `dataset` che sia di questo tipo:
```
{
    "text": [
        <testo 1 formattato con chat template>,
        <testo 2 formattato con chat template>,
        ...
    ]
}
```
cioè popolato da stringhe formattate con il chat template del modello.

Dobbiamo passare dal generico formato di Huggingface `("role", "content")` al formato specifico del modello selezioanto.

In [12]:
# Creiamo la funzione da applicare successivamente al dataset per creare la nuova colonna "text" con i chat template corretti per il modello.
def formatting_prompts_func(data):
    conversations = data["conversations"]
    texts = [tokenizer.apply_chat_template(conversation['messages'], tokenize=False, add_generation_prompt=False) for conversation in conversations]
    return {"text" : texts}

In [13]:
# Applichiamo la funzione al datset
dataset = dataset.map(formatting_prompts_func, batched = True, num_proc=1)

# Ora abbiamo creato correttamente il nuovo campo "text"
print(dataset)
display(dataset.features)

Map: 100%|██████████| 173/173 [00:00<00:00, 17134.16 examples/s]

Dataset({
    features: ['conversations', 'text'],
    num_rows: 173
})


{'conversations': {'messages': [{'content': Value(dtype='string', id=None),
    'role': Value(dtype='string', id=None)}]},
 'text': Value(dtype='string', id=None)}

In [14]:
# Osserviamo ora il primo record del dataset
print(dataset[0]['text'])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Sep 2025

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

RM ADDOME INFERIORE (S/C MDC)


L'ESAME E STATO ESEGUITO MEDINTE SEQUENZE FSE,DWI, MIRATO ALLA STADIAZIONE LOCALE DELLA NEOPLASIA RETTALE .

IN CORRISPONDENZA DELLA PARETE POSTEROLATERALE SINISTRA DEL RETTO MEDIO-BASSO SI SEGNALA FORMAZIONE AGGETTANTE CHE DA CIRCA 5 CM DALLO SFINTERE ANALE INTERNO SI ESTENDE CRANIALMENTE PER CIRCA 5 CM, OCCUPA 2/4 DELLE CIRCONFERENZA DEL LUME , INVIA DIGIRTAZIONI POLIPOIDI NEL MESORETTO POSTERIORE IN CONTINUITA CON DEPOSITI TUMORALI CHE AVVOLGONO A COLATA ED INGLOBANO I VASI EMORROIDARI SUPERIORI FINO AL PROMONTORIO SACRALE E RETRAGGONO LA FASCIA PERIRETTALE -PRESACRALE DALLE ORE 4 ALLE ORE 6.
ADESA ALLA FASCIA PERIRETTALE, ALLE ORE 7-8, E PRESENTE UNA LINFOAD

Vediamo come il chat template ha trasformato le conversazioni.

**[Notice]** Llama 3.1 Instruct's default chat template default adds `"Cutting Knowledge Date: December 2023\nToday Date: 26 July 2024"`, so do not be alarmed!

## Split data

In [15]:
splitted = dataset.train_test_split(test_size=0.2, seed=2025)
print(splitted)

DatasetDict({
    train: Dataset({
        features: ['conversations', 'text'],
        num_rows: 138
    })
    test: Dataset({
        features: ['conversations', 'text'],
        num_rows: 35
    })
})


In [16]:
train_split = splitted['train']
test_split = splitted['test']

<a name="Train"></a>
# Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [28]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_split,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
        dataset_num_proc=1
    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 138/138 [00:00<00:00, 1366.69 examples/s]


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs.

In [29]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

Map (num_proc=12): 100%|██████████| 138/138 [00:29<00:00,  4.67 examples/s]


We verify masking is actually done:

In [30]:
print(trainer.train_dataset[5].keys())
print(tokenizer.decode(trainer.train_dataset[5]["input_ids"]))
print('*--- inizio')
for x, y in zip(trainer.train_dataset[5]['input_ids'][:5], trainer.train_dataset[5]['labels'][:5]):
    print(f"id: {x:<10}label: {y}")
print('\n*--- fine')
for x, y in zip(trainer.train_dataset[5]['input_ids'][-5:], trainer.train_dataset[5]['labels'][-5:]):
    print(f"id: {x:<10}label: {y}")

dict_keys(['conversations', 'text', 'input_ids', 'attention_mask', 'labels'])
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 28 Sep 2025

Sei un modello di intelligenza artificiale specializzato nell'estrazione di informazioni cliniche dai referti radiologici.<|eot_id|><|start_header_id|>user<|end_header_id|>

IN CORRISPONDENZA DELLA PARETE ANTERIORE DEL RETTO MEDIO-ALTO, DA CIRCA 7 CM DALLO SFINTERE ANALE INTERNO CON ESTENSIONE CRANIALE PER CIRCA 2 CM, SI DOCUMENTA ISPESSIMENTO PARIETALE CONFINATO ALLA PARETE (SPESSORE MASSIMO 13 MM), CHE OCCUPA DUE QUARTI DELLA CIRCONFERENZA DEL LUME, MODERATAMENTE IPERINTENSO NELLE SEQUENZE T2-DIPENDENTI ED IPERINTENSO NELLE SEQUENZE DWI, CHE MOSTRA INTENSA IMPREGNAZIONE CONSTRASTOGRAFICA NELLE SEQUENZE DI PERFUSIONE. 
DUE LINFONODI CON ASSE CORTO <5 MM IN SEDE MESORETTALE POSTERIORE, ASPECIFICI. NON EVIDENTI LINFOADENOPATIE IN SEDE PELVICA. 
PICCOLE CISTI DI TARLOV A L

In [31]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]  # id di sapce
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

'                                                                                                                                                                                                                                                                                                                                 {"morfologia": "mucinoso", "spessore_parietale": 13.0, "estensione_cranio_caudale": 20.0, "distanza_oai": 70.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": null, "infiltrazione_tessuto_adiposo": "no", "infiltrazione_sfinteri": "no", "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "no", "linfonodi_sospetti": 0.0, "sedi_locoregionali": [], "sedi_non_locoregionali": [], "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "no"}<|eot_id|>'

We can see the System and Instruction prompts are successfully masked!

In [32]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5070. Max memory = 11.94 GB.
3.168 GB of memory reserved.


In [33]:
# @title Training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 138 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.290300
2,2.269900
3,2.338500
4,2.270200
5,2.163400
6,1.941800
7,1.656300
8,1.462900
9,1.255700
10,1.082500


In [35]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

266.7409 seconds used for training.
4.45 minutes used for training.
Peak reserved memory = 6.355 GB.
Peak reserved memory for training = 3.187 GB.
Peak reserved memory % of max memory = 53.224 %.
Peak reserved memory for training % of max memory = 26.692 %.


<a name="Inference"></a>
# Inference
Let's run the model! You can change the instruction and input - leave the output blank!



We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [36]:
test_messages = test_split[0]['conversations']['messages'][:2]
tokenized_chat_template = tokenizer.apply_chat_template(
                        test_messages,
                        tokenize = True,
                        add_generation_prompt = True, # Must add for generation
                        return_tensors = "pt"
                        ).to("cuda")

In [37]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

outputs = model.generate(input_ids=tokenized_chat_template,
                         use_cache=True,
                         temperature=1.5,
                         min_p=0.1)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [38]:
char_lenght_inputs = len(tokenizer.decode(tokenized_chat_template[0]))
output_string = tokenizer.batch_decode(outputs)[0][char_lenght_inputs:-10]  # togliere <|eot_id|>
output_string_dict = json.loads(output_string)
display(output_string_dict)

{'morfologia': 'solido_anulare',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 10.0,
 'distanza_oai': None,
 'posizione': 'medio',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no',
 'riflessione_peritoneale_anteriore': 'cavallo',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'sospetto',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': 'no',
 'coinvolgimento_fascia_mesorettale': 'sospetto',
 'linfonodi_sospetti': 0.0,
 'sedi_locoregionali': [],
 'sedi_non_locoregionali': ['otturatori', 'iliaci_esterni'],
 'depositi_tumorali': None,
 'numero_depositi': 0.0,
 'emvi_esteso': 'no'}

In [39]:
groundtruth = test_split[0]['conversations']['messages'][2]['content']
display(json.loads(groundtruth))

{'morfologia': 'solido_polipoide',
 'spessore_parietale': None,
 'estensione_cranio_caudale': 100.0,
 'distanza_oai': None,
 'posizione': 'medio',
 'carcinosi_peritoneale': 'no',
 'lesioni_ossee': 'no',
 'riflessione_peritoneale_anteriore': 'cavallo',
 'infiltrazione_tessuto_adiposo': 'si_5mm_plus',
 'infiltrazione_sfinteri': 'no',
 'infiltrazione_organi_extra': 'no',
 'infiltrazione_organi_dettagli': None,
 'coinvolgimento_riflessione_peritoneale': None,
 'coinvolgimento_fascia_mesorettale': 'rischio',
 'linfonodi_sospetti': 0.0,
 'sedi_locoregionali': ['mesorettali', 'rettali_superiori', 'otturatori'],
 'sedi_non_locoregionali': ['iliaci_esterni'],
 'depositi_tumorali': 'no',
 'numero_depositi': 0.0,
 'emvi_esteso': 'sospetto'}

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=tokenized_chat_template,
                   streamer=text_streamer,
                   use_cache=True,
                   temperature=1.5,
                   min_p=0.1)

{"morfologia": "mucinoso", "spessore_parietale": 75.0, "estensione_cranio_caudale": 0.0, "distanza_oai": 0.0, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "sotto", "infiltrazione_tessuto_adiposo": "si_5mm_plus", "infiltrazione_sfinteri": null, "infiltrazione_organi_extra": "no", "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "no", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 4.0, "sedi_locoregionali": ["otturatori"], "sedi_non_locoregionali": [], "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "si"}<|eot_id|>


<a name="Save"></a>
# Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained(saving_name)  # Local saving
tokenizer.save_pretrained(saving_name)
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('Llama-3.2-3B-Instruct-lora\\tokenizer_config.json',
 'Llama-3.2-3B-Instruct-lora\\special_tokens_map.json',
 'Llama-3.2-3B-Instruct-lora\\chat_template.jinja',
 'Llama-3.2-3B-Instruct-lora\\tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [20]:
if True:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = saving_name, # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

test_messages = test_split[0]['conversations']['messages'][:2]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize = True,
    add_generation_prompt = True, # Must add for generation
    return_tensors = "pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids = inputs, streamer = text_streamer,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

==((====))==  Unsloth 2025.9.7: Fast Llama patching. Transformers: 4.55.2.
   \\   /|    NVIDIA GeForce RTX 5070. Num GPUs = 1. Max memory: 11.94 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 12.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


{"morfologia": "mucinoso", "spessore_parietale": null, "estensione_cranio_caudale": 100.0, "distanza_oai": null, "posizione": "medio", "carcinosi_peritoneale": "no", "lesioni_ossee": "no", "riflessione_peritoneale_anteriore": "sotto", "infiltrazione_tessuto_adiposo": "si_5mm", "infiltrazione_sfinteri": null, "infiltrazione_organi_extra": null, "infiltrazione_organi_dettagli": null, "coinvolgimento_riflessione_peritoneale": "si", "coinvolgimento_fascia_mesorettale": "si", "linfonodi_sospetti": 7.0, "sedi_locoregionali": ["mesorettali", "rettali_superiori", "otturatori", "inguinali"], "sedi_non_locoregionali": [], "depositi_tumorali": "no", "numero_depositi": 0.0, "emvi_esteso": "no"}<|eot_id|>


You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.

# Alternative di salvataggio e caricamento

In [ ]:
if False:
    # I highly do NOT suggest - use Unsloth if possible
    from peft import AutoPeftModelForCausalLM
    from transformers import AutoTokenizer
    model = AutoPeftModelForCausalLM.from_pretrained(
        "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = load_in_4bit,
    )
    tokenizer = AutoTokenizer.from_pretrained("lora_model")

### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "", # Get a token at https://huggingface.co/settings/tokens
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>
